# Objective 3 — Step 7: German Heterogeneous Ensemble Development

This notebook develops the **hybrid supervised ensemble architecture on German Credit only**.

## Evidence carried forward from Steps 4–6

- Group-aware Chi-Square Top-75% is the frozen feature-selection candidate.
- Feature selection is not forced universally because external replication was dataset-dependent.
- Standard balanced class weighting is retained as the principled imbalance-aware option.
- Cost2/Cost5 weighting is not used in the ensemble because those multipliers impose stronger assumed cost preferences and showed over-correction on some datasets.

## Base learners

Three heterogeneous supervised learners are combined:

1. Logistic Regression — linear probabilistic learner
2. Random Forest — bagging/tree learner
3. XGBoost — boosted nonlinear learner

SVM is not included in the main stack because Step 3 showed much higher computation time on the 30,000-record Taiwan dataset without a corresponding overall performance advantage.

## Ensemble candidates

For each feature regime and training regime:

- Equal-probability Soft Voting
- Out-of-fold Logistic-Regression Stacking

Feature regimes:
- All Features
- Frozen Group-Aware Chi-Square Top-75%

Training regimes:
- Unweighted
- Balanced

This gives **8 ensemble configurations**.

## Leakage control

For stacking, the meta-learner is trained only on **inner out-of-fold base predictions**.  
When feature selection is used, Chi-Square ranking is re-fitted inside every inner-training fold before generating inner-validation predictions.

The outer test fold is never used for preprocessing, feature selection, weighting, base-model training, or meta-model training.


In [1]:
%pip install pandas numpy scikit-learn xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2
[notice] To update, run: C:\Users\hp\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [2]:

from pathlib import Path
import json
import math
import time
import warnings

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import chi2
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

BASE_DIR = Path(r"D:\PHD\Research Paper writing\3rd Obj. paper")

STEP2_DIR = BASE_DIR / "results" / "preprocessing_protocol"
BASELINE_DIR = BASE_DIR / "results" / "baseline_models"
STEP6_DIR = BASE_DIR / "results" / "cost_sensitive_xgb_ablation"
DATA_DIR = BASE_DIR / "data" / "processed"

OUT_DIR = BASE_DIR / "results" / "ensemble_german_development"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GERMAN_FILE = DATA_DIR / "german_credit_cleaned.csv"
DICTIONARY_FILE = STEP2_DIR / "preprocessing_data_dictionary.csv"
BASELINE_PREDICTIONS_FILE = BASELINE_DIR / "baseline_predictions_all.csv"
STEP6_RESULTS_FILE = STEP6_DIR / "cost_sensitive_xgb_fold_results.csv"

required = [
    GERMAN_FILE,
    DICTIONARY_FILE,
    BASELINE_PREDICTIONS_FILE,
    STEP6_RESULTS_FILE,
]

missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Required previous-step files are missing:\n" + "\n".join(missing)
    )

OUTER_REPEAT_SEEDS = [42, 142, 242, 342, 442]
OUTER_FOLDS = 5
INNER_FOLDS = 5
FROZEN_FRACTION = 0.75

FEATURE_REGIMES = ["AllFeatures", "FrozenChi2Top75"]
TRAINING_REGIMES = ["Unweighted", "Balanced"]
ENSEMBLE_TYPES = ["SoftVote", "Stack"]

BASE_MODELS = ["LR", "RF", "XGB"]

print("Output folder:", OUT_DIR)


Output folder: D:\PHD\Research Paper writing\3rd Obj. paper\results\ensemble_german_development


## 1. Load German Credit and recover feature roles

In [3]:

df = pd.read_csv(GERMAN_FILE)
dictionary = pd.read_csv(DICTIONARY_FILE)
baseline_predictions = pd.read_csv(BASELINE_PREDICTIONS_FILE)
step6_results = pd.read_csv(STEP6_RESULTS_FILE)

dataset_name = "German Credit"

d = dictionary[dictionary["dataset"] == dataset_name].copy()

categorical_features = (
    d.loc[d["role"] == "categorical", "variable"]
    .astype(str).tolist()
)
ordinal_features = (
    d.loc[d["role"] == "ordinal", "variable"]
    .astype(str).tolist()
)
numerical_features = (
    d.loc[d["role"] == "numerical", "variable"]
    .astype(str).tolist()
)

all_features = (
    categorical_features
    + ordinal_features
    + numerical_features
)

X = df[all_features].copy()
y = df["adverse_target"].astype(int).copy()
groups = df["profile_group_id"].astype(str).copy()

print("Records:", len(df))
print("Predictors:", len(all_features))
print("Adverse cases:", int(y.sum()))
print("Adverse rate:", round(y.mean(), 4))

assert len(all_features) == 20
assert set(y.unique()).issubset({0, 1})


Records: 1000
Predictors: 20
Adverse cases: 300
Adverse rate: 0.3


## 2. Preprocessing and group-aware Chi-Square functions

In [4]:

def make_one_hot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
            dtype=np.float32,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
            dtype=np.float32,
        )


def build_preprocessor(mode, selected_features):
    selected_features = list(selected_features)

    selected_cat = [
        f for f in categorical_features
        if f in selected_features
    ]
    selected_ord = [
        f for f in ordinal_features
        if f in selected_features
    ]
    selected_num = [
        f for f in numerical_features
        if f in selected_features
    ]

    transformers = []

    if selected_num:
        if mode == "scaled":
            num_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ])
        elif mode == "tree":
            num_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ])
        elif mode == "chi2":
            num_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", MinMaxScaler(clip=True)),
            ])
        else:
            raise ValueError(mode)

        transformers.append(("num", num_pipe, selected_num))

    if selected_ord:
        if mode == "scaled":
            ord_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("scaler", StandardScaler()),
            ])
        elif mode == "tree":
            ord_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
            ])
        elif mode == "chi2":
            ord_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("scaler", MinMaxScaler(clip=True)),
            ])
        else:
            raise ValueError(mode)

        transformers.append(("ord", ord_pipe, selected_ord))

    if selected_cat:
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ])
        transformers.append(("cat", cat_pipe, selected_cat))

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=True,
    )


def feature_map_from_fitted_chi2_preprocessor(prep):
    rows = []
    idx = 0

    for feature in numerical_features:
        rows.append({
            "transformed_index": idx,
            "source_feature": feature,
            "source_role": "numerical",
        })
        idx += 1

    for feature in ordinal_features:
        rows.append({
            "transformed_index": idx,
            "source_feature": feature,
            "source_role": "ordinal",
        })
        idx += 1

    if categorical_features:
        cat_pipe = prep.named_transformers_["cat"]
        encoder = cat_pipe.named_steps["onehot"]

        for source_feature, categories in zip(
            categorical_features,
            encoder.categories_,
        ):
            for _ in categories:
                rows.append({
                    "transformed_index": idx,
                    "source_feature": source_feature,
                    "source_role": "categorical",
                })
                idx += 1

    fmap = pd.DataFrame(rows)

    assert len(fmap) == len(prep.get_feature_names_out())
    return fmap


def group_aware_chi2_top75(X_train, y_train):
    prep = build_preprocessor(
        "chi2",
        all_features,
    )

    X_chi = prep.fit_transform(
        X_train[all_features],
        y_train,
    )

    assert np.asarray(X_chi).min() >= -1e-12

    fmap = feature_map_from_fitted_chi2_preprocessor(prep)
    raw_scores, _ = chi2(X_chi, y_train)

    temp = fmap.copy()
    temp["raw_score"] = (
        pd.Series(raw_scores)
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .to_numpy()
    )

    n = len(temp)
    ranks = temp["raw_score"].rank(
        ascending=False,
        method="average",
    )

    temp["normalized_relevance"] = (
        1.0 - (ranks - 1.0) / (n - 1.0)
        if n > 1
        else 1.0
    )

    grouped = (
        temp.groupby("source_feature", as_index=False)
        .agg(group_score=("normalized_relevance", "mean"))
    )

    grouped["source_rank"] = grouped["group_score"].rank(
        ascending=False,
        method="average",
    )

    grouped = grouped.sort_values(
        ["source_rank", "source_feature"]
    ).reset_index(drop=True)

    n_select = int(
        math.ceil(len(grouped) * FROZEN_FRACTION)
    )

    return grouped["source_feature"].astype(str).tolist()[:n_select]


## 3. Base-model builders

In [5]:

def training_ratio(y_train):
    n_pos = int((y_train == 1).sum())
    n_neg = int((y_train == 0).sum())
    return n_neg / n_pos


def build_base_pipeline(
    model_name,
    selected_features,
    training_regime,
    y_training_partition,
):
    balanced = training_regime == "Balanced"

    if model_name == "LR":
        estimator = LogisticRegression(
            max_iter=3000,
            solver="lbfgs",
            class_weight="balanced" if balanced else None,
            random_state=42,
        )
        mode = "scaled"

    elif model_name == "RF":
        estimator = RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced" if balanced else None,
            random_state=42,
            n_jobs=-1,
        )
        mode = "tree"

    elif model_name == "XGB":
        spw = (
            training_ratio(y_training_partition)
            if balanced
            else 1.0
        )

        estimator = XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=float(spw),
            random_state=42,
            n_jobs=-1,
            verbosity=0,
        )
        mode = "tree"

    else:
        raise ValueError(model_name)

    return Pipeline([
        (
            "preprocessor",
            build_preprocessor(
                mode,
                selected_features,
            ),
        ),
        ("model", estimator),
    ])


def predict_positive_probability(pipe, X_part):
    return pipe.predict_proba(X_part)[:, 1]


## 4. Metrics

In [6]:

def calculate_metrics(y_true, y_pred, y_score):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )
    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    gmean = (
        math.sqrt(specificity * sensitivity)
        if not np.isnan(specificity + sensitivity)
        else np.nan
    )

    fpr, tpr, _ = roc_curve(y_true, y_score)
    ks = float(np.max(tpr - fpr))

    n = len(y_true)

    costs = {}
    for fn_cost in [1, 2, 5, 10]:
        total_cost = fp + fn_cost * fn
        costs[f"cost_FN{fn_cost}_FP1_per100"] = (
            100.0 * total_cost / n
        )

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_adverse": precision_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "recall_adverse": recall_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "specificity": specificity,
        "f1_adverse": f1_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_score),
        "pr_auc": average_precision_score(y_true, y_score),
        "gmean": gmean,
        "ks_statistic": ks,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        **costs,
    }


## 5. Recreate and verify the exact Step-3 outer folds

In [7]:

saved_step3_xgb = baseline_predictions[
    (baseline_predictions["dataset"] == dataset_name)
    & (baseline_predictions["model"] == "XGB")
].copy()

outer_splits = {}

for repeat_no, seed in enumerate(
    OUTER_REPEAT_SEEDS,
    start=1,
):
    splitter = StratifiedGroupKFold(
        n_splits=OUTER_FOLDS,
        shuffle=True,
        random_state=seed,
    )

    for fold_no, (train_idx, test_idx) in enumerate(
        splitter.split(X, y, groups),
        start=1,
    ):
        run_id = f"R{repeat_no}_F{fold_no}"

        expected_test = set(
            np.asarray(test_idx, dtype=int).tolist()
        )

        saved_test = set(
            saved_step3_xgb.loc[
                saved_step3_xgb["run_id"] == run_id,
                "source_row_index",
            ].astype(int).tolist()
        )

        assert expected_test == saved_test

        assert len(
            set(groups.iloc[train_idx]).intersection(
                set(groups.iloc[test_idx])
            )
        ) == 0

        outer_splits[run_id] = {
            "repeat": repeat_no,
            "fold": fold_no,
            "seed": seed,
            "train_idx": np.asarray(train_idx, dtype=int),
            "test_idx": np.asarray(test_idx, dtype=int),
        }

print("All 25 outer folds exactly match Step 3.")


All 25 outer folds exactly match Step 3.


## 6. Leakage-free ensemble evaluation function

For `FrozenChi2Top75`, feature selection is performed:

- separately inside each inner-training fold for OOF predictions;
- once again on the full outer-training fold before predicting the outer test fold.

This avoids using inner-validation labels during feature selection.


In [8]:

def evaluate_outer_ensemble(
    run_id,
    info,
    feature_regime,
    training_regime,
):
    train_idx = info["train_idx"]
    test_idx = info["test_idx"]

    X_outer_train = X.iloc[train_idx].copy()
    X_outer_test = X.iloc[test_idx].copy()

    y_outer_train = y.iloc[train_idx].copy()
    y_outer_test = y.iloc[test_idx].copy()

    g_outer_train = groups.iloc[train_idx].copy()

    n_train = len(X_outer_train)

    # OOF probability matrix for three base learners.
    oof_matrix = np.full(
        (n_train, len(BASE_MODELS)),
        np.nan,
        dtype=float,
    )

    inner_seed = 10000 + info["seed"] + info["fold"]

    inner_splitter = StratifiedGroupKFold(
        n_splits=INNER_FOLDS,
        shuffle=True,
        random_state=inner_seed,
    )

    inner_fs_records = []

    for inner_fold, (
        inner_train_pos,
        inner_valid_pos,
    ) in enumerate(
        inner_splitter.split(
            X_outer_train,
            y_outer_train,
            g_outer_train,
        ),
        start=1,
    ):
        X_inner_train = X_outer_train.iloc[
            inner_train_pos
        ].copy()

        X_inner_valid = X_outer_train.iloc[
            inner_valid_pos
        ].copy()

        y_inner_train = y_outer_train.iloc[
            inner_train_pos
        ].copy()

        if feature_regime == "AllFeatures":
            selected_inner = all_features

        elif feature_regime == "FrozenChi2Top75":
            selected_inner = group_aware_chi2_top75(
                X_inner_train,
                y_inner_train,
            )

        else:
            raise ValueError(feature_regime)

        inner_fs_records.append({
            "run_id": run_id,
            "inner_fold": inner_fold,
            "feature_regime": feature_regime,
            "training_regime": training_regime,
            "selected_source_features": len(selected_inner),
            "selected_features": ";".join(selected_inner),
        })

        for model_position, model_name in enumerate(
            BASE_MODELS
        ):
            pipe = build_base_pipeline(
                model_name=model_name,
                selected_features=selected_inner,
                training_regime=training_regime,
                y_training_partition=y_inner_train,
            )

            pipe.fit(
                X_inner_train[selected_inner],
                y_inner_train,
            )

            oof_matrix[
                inner_valid_pos,
                model_position,
            ] = predict_positive_probability(
                pipe,
                X_inner_valid[selected_inner],
            )

    assert np.isfinite(oof_matrix).all()

    # Final feature selection on full outer-training data.
    if feature_regime == "AllFeatures":
        selected_outer = all_features
    else:
        selected_outer = group_aware_chi2_top75(
            X_outer_train,
            y_outer_train,
        )

    outer_test_base_probs = np.zeros(
        (len(X_outer_test), len(BASE_MODELS)),
        dtype=float,
    )

    final_base_runtime = 0.0

    for model_position, model_name in enumerate(
        BASE_MODELS
    ):
        pipe = build_base_pipeline(
            model_name=model_name,
            selected_features=selected_outer,
            training_regime=training_regime,
            y_training_partition=y_outer_train,
        )

        start = time.perf_counter()

        pipe.fit(
            X_outer_train[selected_outer],
            y_outer_train,
        )

        outer_test_base_probs[
            :,
            model_position,
        ] = predict_positive_probability(
            pipe,
            X_outer_test[selected_outer],
        )

        final_base_runtime += (
            time.perf_counter() - start
        )

    # ------------------------------------
    # Equal soft voting
    # ------------------------------------
    soft_score = outer_test_base_probs.mean(axis=1)
    soft_pred = (soft_score >= 0.5).astype(int)

    soft_metrics = calculate_metrics(
        y_outer_test,
        soft_pred,
        soft_score,
    )

    # ------------------------------------
    # OOF logistic-regression stacking
    # ------------------------------------
    meta = LogisticRegression(
        max_iter=2000,
        solver="lbfgs",
        class_weight=(
            "balanced"
            if training_regime == "Balanced"
            else None
        ),
        random_state=42,
    )

    meta_start = time.perf_counter()

    meta.fit(
        oof_matrix,
        y_outer_train,
    )

    stack_score = meta.predict_proba(
        outer_test_base_probs
    )[:, 1]

    stack_pred = (
        stack_score >= 0.5
    ).astype(int)

    meta_runtime = (
        time.perf_counter() - meta_start
    )

    stack_metrics = calculate_metrics(
        y_outer_test,
        stack_pred,
        stack_score,
    )

    # Save prediction-level rows for later calibration.
    prediction_rows = []

    for local_position, source_idx in enumerate(test_idx):
        base_values = {
            f"base_{BASE_MODELS[j]}_prob": float(
                outer_test_base_probs[
                    local_position,
                    j,
                ]
            )
            for j in range(
                len(BASE_MODELS)
            )
        }

        prediction_rows.append({
            "dataset": dataset_name,
            "run_id": run_id,
            "repeat": info["repeat"],
            "fold": info["fold"],
            "feature_regime": feature_regime,
            "training_regime": training_regime,
            "source_row_index": int(source_idx),
            "y_true": int(
                y_outer_test.iloc[
                    local_position
                ]
            ),
            "softvote_score": float(
                soft_score[local_position]
            ),
            "stack_score": float(
                stack_score[local_position]
            ),
            **base_values,
        })

    result_rows = []

    common = {
        "dataset": dataset_name,
        "run_id": run_id,
        "repeat": info["repeat"],
        "fold": info["fold"],
        "feature_regime": feature_regime,
        "training_regime": training_regime,
        "selected_outer_source_features": len(
            selected_outer
        ),
        "selected_outer_features": ";".join(
            selected_outer
        ),
        "base_model_runtime_seconds": (
            final_base_runtime
        ),
        "meta_runtime_seconds": (
            meta_runtime
        ),
    }

    result_rows.append({
        **common,
        "ensemble_type": "SoftVote",
        **soft_metrics,
    })

    result_rows.append({
        **common,
        "ensemble_type": "Stack",
        **stack_metrics,
    })

    meta_coefficients = {
        "run_id": run_id,
        "feature_regime": feature_regime,
        "training_regime": training_regime,
        "meta_intercept": float(
            meta.intercept_[0]
        ),
        **{
            f"coef_{BASE_MODELS[j]}": float(
                meta.coef_[0, j]
            )
            for j in range(
                len(BASE_MODELS)
            )
        },
    }

    return (
        result_rows,
        prediction_rows,
        inner_fs_records,
        meta_coefficients,
    )


## 7. Run German ensemble development

Expected ensemble evaluations:

`25 outer runs × 2 feature regimes × 2 training regimes × 2 ensemble types = 200`


In [9]:

all_results = []
all_predictions = []
all_inner_fs = []
all_meta_coefficients = []

for run_number, (run_id, info) in enumerate(
    outer_splits.items(),
    start=1,
):
    print("\n" + "=" * 80)
    print(f"{run_id} ({run_number}/25)")
    print("=" * 80)

    for feature_regime in FEATURE_REGIMES:
        for training_regime in TRAINING_REGIMES:
            start = time.perf_counter()

            (
                result_rows,
                prediction_rows,
                inner_fs_records,
                meta_coefficients,
            ) = evaluate_outer_ensemble(
                run_id=run_id,
                info=info,
                feature_regime=feature_regime,
                training_regime=training_regime,
            )

            elapsed = time.perf_counter() - start

            all_results.extend(result_rows)
            all_predictions.extend(prediction_rows)
            all_inner_fs.extend(inner_fs_records)
            all_meta_coefficients.append(
                meta_coefficients
            )

            stack_row = [
                row
                for row in result_rows
                if row["ensemble_type"] == "Stack"
            ][0]

            print(
                f"{feature_regime:16s} | "
                f"{training_regime:10s} | "
                f"Stack ROC={stack_row['roc_auc']:.4f} | "
                f"F1={stack_row['f1_adverse']:.4f} | "
                f"MCC={stack_row['mcc']:.4f} | "
                f"elapsed={elapsed:.2f}s"
            )


results = pd.DataFrame(all_results)
predictions = pd.DataFrame(all_predictions)
inner_fs = pd.DataFrame(all_inner_fs)
meta_coefficients = pd.DataFrame(all_meta_coefficients)

results.to_csv(
    OUT_DIR / "german_ensemble_fold_results.csv",
    index=False,
)

predictions.to_csv(
    OUT_DIR / "german_ensemble_outer_predictions.csv",
    index=False,
)

inner_fs.to_csv(
    OUT_DIR / "german_ensemble_inner_feature_sets.csv",
    index=False,
)

meta_coefficients.to_csv(
    OUT_DIR / "german_stack_meta_coefficients.csv",
    index=False,
)

print("\nGerman ensemble development completed.")



R1_F1 (1/25)
AllFeatures      | Unweighted | Stack ROC=0.8431 | F1=0.6250 | MCC=0.5055 | elapsed=5.88s
AllFeatures      | Balanced   | Stack ROC=0.8456 | F1=0.6871 | MCC=0.5055 | elapsed=5.82s
FrozenChi2Top75  | Unweighted | Stack ROC=0.8519 | F1=0.6195 | MCC=0.4931 | elapsed=5.94s
FrozenChi2Top75  | Balanced   | Stack ROC=0.8527 | F1=0.6585 | MCC=0.4558 | elapsed=5.94s

R1_F2 (2/25)
AllFeatures      | Unweighted | Stack ROC=0.7681 | F1=0.4878 | MCC=0.3690 | elapsed=5.85s
AllFeatures      | Balanced   | Stack ROC=0.7564 | F1=0.5410 | MCC=0.3695 | elapsed=5.90s
FrozenChi2Top75  | Unweighted | Stack ROC=0.7741 | F1=0.5116 | MCC=0.3844 | elapsed=5.99s
FrozenChi2Top75  | Balanced   | Stack ROC=0.7725 | F1=0.5760 | MCC=0.4215 | elapsed=5.94s

R1_F3 (3/25)
AllFeatures      | Unweighted | Stack ROC=0.7991 | F1=0.5545 | MCC=0.4398 | elapsed=5.97s
AllFeatures      | Balanced   | Stack ROC=0.7957 | F1=0.6232 | MCC=0.4308 | elapsed=6.10s
FrozenChi2Top75  | Unweighted | Stack ROC=0.7937 | F1=0.50

## 8. Ensemble summary

In [10]:

summary = (
    results
    .groupby(
        [
            "feature_regime",
            "training_regime",
            "ensemble_type",
        ],
        as_index=False,
    )
    .agg(
        Source_Features=("selected_outer_source_features", "mean"),
        ROC_AUC=("roc_auc", "mean"),
        ROC_AUC_SD=("roc_auc", "std"),
        PR_AUC=("pr_auc", "mean"),
        PR_AUC_SD=("pr_auc", "std"),
        Recall=("recall_adverse", "mean"),
        Precision=("precision_adverse", "mean"),
        F1=("f1_adverse", "mean"),
        Balanced_Accuracy=("balanced_accuracy", "mean"),
        MCC=("mcc", "mean"),
        MCC_SD=("mcc", "std"),
        GMean=("gmean", "mean"),
        KS=("ks_statistic", "mean"),
        Cost_2_1=("cost_FN2_FP1_per100", "mean"),
        Cost_5_1=("cost_FN5_FP1_per100", "mean"),
        Cost_10_1=("cost_FN10_FP1_per100", "mean"),
    )
)

summary.to_csv(
    OUT_DIR / "german_ensemble_summary.csv",
    index=False,
)

display(
    summary.sort_values(
        ["MCC", "Balanced_Accuracy", "ROC_AUC"],
        ascending=False,
    )
)


,feature_regime,training_regime,ensemble_type,Source_Features,ROC_AUC,ROC_AUC_SD,PR_AUC,PR_AUC_SD,Recall,Precision,F1,Balanced_Accuracy,MCC,MCC_SD,GMean,KS,Cost_2_1,Cost_5_1,Cost_10_1
4,FrozenChi2Top75,Balanced,SoftVote,15.0,0.798802,0.025750,0.634897,0.054347,0.626506,0.586505,0.603075,0.718355,0.428590,0.052074,0.711347,0.494290,35.78,69.50,125.70
0,AllFeatures,Balanced,SoftVote,20.0,0.800288,0.026113,0.650283,0.058284,0.603789,0.588792,0.593839,0.711174,0.419291,0.053545,0.701991,0.493052,36.50,72.20,131.70
5,FrozenChi2Top75,Balanced,Stack,15.0,0.797647,0.027811,0.631346,0.057007,0.723905,0.525251,0.606986,0.721884,0.412758,0.046625,0.720647,0.493595,36.18,61.08,102.58
1,AllFeatures,Balanced,Stack,20.0,0.798961,0.027562,0.648493,0.058495,0.712393,0.526961,0.603991,0.719247,0.409006,0.053748,0.718214,0.489844,36.44,62.36,105.56
6,FrozenChi2Top75,Unweighted,SoftVote,15.0,0.799230,0.024581,0.633553,0.053496,0.464858,0.659474,0.541201,0.680995,0.405399,0.055008,0.643648,0.496043,39.42,87.72,168.22
2,AllFeatures,Unweighted,SoftVote,20.0,0.800488,0.024706,0.647968,0.059295,0.457942,0.658284,0.536641,0.678098,0.400987,0.069922,0.639252,0.494153,39.70,88.54,169.94
7,FrozenChi2Top75,Unweighted,Stack,15.0,0.798378,0.026055,0.631874,0.056882,0.439599,0.663388,0.524543,0.671996,0.394160,0.057434,0.627978,0.491615,40.38,90.90,175.10
3,AllFeatures,Unweighted,Stack,20.0,0.799880,0.025107,0.648672,0.057668,0.443064,0.660657,0.526711,0.672643,0.394129,0.071652,0.630011,0.490087,40.30,90.46,174.06


## 9. Add Step-6 XGBoost references to the German decision table

In [11]:

reference_xgb = step6_results[
    (
        step6_results["dataset"] == dataset_name
    )
    & (
        step6_results["feature_regime"].isin(
            FEATURE_REGIMES
        )
    )
    & (
        step6_results["weight_strategy"].isin(
            TRAINING_REGIMES
        )
    )
].copy()

reference_summary = (
    reference_xgb
    .groupby(
        [
            "feature_regime",
            "weight_strategy",
        ],
        as_index=False,
    )
    .agg(
        Source_Features=("source_features", "mean"),
        ROC_AUC=("roc_auc", "mean"),
        PR_AUC=("pr_auc", "mean"),
        Recall=("recall_adverse", "mean"),
        Precision=("precision_adverse", "mean"),
        F1=("f1_adverse", "mean"),
        Balanced_Accuracy=("balanced_accuracy", "mean"),
        MCC=("mcc", "mean"),
        GMean=("gmean", "mean"),
        KS=("ks_statistic", "mean"),
        Cost_2_1=("cost_FN2_FP1_per100", "mean"),
        Cost_5_1=("cost_FN5_FP1_per100", "mean"),
        Cost_10_1=("cost_FN10_FP1_per100", "mean"),
    )
)

reference_summary["training_regime"] = (
    reference_summary["weight_strategy"]
)

reference_summary["ensemble_type"] = "Single_XGB"

reference_summary = reference_summary.drop(
    columns=["weight_strategy"]
)

decision_table = pd.concat(
    [
        reference_summary,
        summary.drop(
            columns=[
                "ROC_AUC_SD",
                "PR_AUC_SD",
                "MCC_SD",
            ]
        ),
    ],
    ignore_index=True,
    sort=False,
)

decision_table.to_csv(
    OUT_DIR / "german_ensemble_decision_table.csv",
    index=False,
)

display(
    decision_table.sort_values(
        ["MCC", "Balanced_Accuracy", "ROC_AUC"],
        ascending=False,
    )
)


,feature_regime,Source_Features,ROC_AUC,PR_AUC,Recall,Precision,F1,Balanced_Accuracy,MCC,GMean,KS,Cost_2_1,Cost_5_1,Cost_10_1,training_regime,ensemble_type
8,FrozenChi2Top75,15.0,0.798802,0.634897,0.626506,0.586505,0.603075,0.718355,0.428590,0.711347,0.494290,35.78,69.50,125.70,Balanced,SoftVote
4,AllFeatures,20.0,0.800288,0.650283,0.603789,0.588792,0.593839,0.711174,0.419291,0.701991,0.493052,36.50,72.20,131.70,Balanced,SoftVote
2,FrozenChi2Top75,15.0,0.787275,0.619312,0.641251,0.565414,0.598078,0.714983,0.415944,0.709594,0.485243,36.32,68.60,122.40,Balanced,Single_XGB
9,FrozenChi2Top75,15.0,0.797647,0.631346,0.723905,0.525251,0.606986,0.721884,0.412758,0.720647,0.493595,36.18,61.08,102.58,Balanced,Stack
5,AllFeatures,20.0,0.798961,0.648493,0.712393,0.526961,0.603991,0.719247,0.409006,0.718214,0.489844,36.44,62.36,105.56,Balanced,Stack
10,FrozenChi2Top75,15.0,0.799230,0.633553,0.464858,0.659474,0.541201,0.680995,0.405399,0.643648,0.496043,39.42,87.72,168.22,Unweighted,SoftVote
0,AllFeatures,20.0,0.790088,0.632007,0.608736,0.569841,0.586398,0.705714,0.404344,0.697714,0.477794,37.18,72.22,130.62,Balanced,Single_XGB
6,AllFeatures,20.0,0.800488,0.647968,0.457942,0.658284,0.536641,0.678098,0.400987,0.639252,0.494153,39.70,88.54,169.94,Unweighted,SoftVote
1,AllFeatures,20.0,0.790369,0.632366,0.478338,0.640146,0.544334,0.681676,0.398821,0.648509,0.478771,39.30,86.16,164.26,Unweighted,Single_XGB
3,FrozenChi2Top75,15.0,0.788434,0.622283,0.490576,0.632947,0.547579,0.683953,0.398727,0.653594,0.485773,39.22,85.12,161.62,Unweighted,Single_XGB


## 10. Final consistency checks

The final ensemble architecture will be selected **after reviewing German results only**.  
Australian and Taiwan will be used later for frozen replication.


In [12]:

expected_rows = (
    25
    * len(FEATURE_REGIMES)
    * len(TRAINING_REGIMES)
    * len(ENSEMBLE_TYPES)
)

assert len(results) == expected_rows, (
    f"Expected {expected_rows} rows; found {len(results)}."
)

assert results["roc_auc"].between(0, 1).all()
assert results["pr_auc"].between(0, 1).all()
assert results["mcc"].between(-1, 1).all()

assert not results[
    [
        "roc_auc",
        "pr_auc",
        "recall_adverse",
        "f1_adverse",
        "balanced_accuracy",
        "mcc",
    ]
].isna().any().any()

configuration = {
    "stage": (
        "Objective 3 Step 7 - German heterogeneous ensemble development"
    ),
    "development_dataset": "German Credit",
    "base_models": BASE_MODELS,
    "feature_regimes": FEATURE_REGIMES,
    "training_regimes": TRAINING_REGIMES,
    "ensemble_types": ENSEMBLE_TYPES,
    "outer_validation": "same 5x5 grouped folds as Steps 3-6",
    "inner_validation_for_stacking": "5-fold StratifiedGroupKFold",
    "feature_selection_rule": (
        "Frozen Group-Aware Chi-Square Top75; refitted inside each inner fold "
        "when producing stacking OOF predictions"
    ),
    "balanced_training": {
        "LR": "class_weight='balanced'",
        "RF": "class_weight='balanced'",
        "XGB": "scale_pos_weight=N_negative/N_positive from current training partition",
        "stack_meta_LR": "class_weight='balanced'",
    },
    "selection_rule": (
        "Choose ensemble configuration using German evidence only. "
        "External datasets must not influence architecture selection."
    ),
}

with open(
    OUT_DIR / "step7_experiment_configuration.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(configuration, f, indent=4)

manifest = sorted(
    [p.name for p in OUT_DIR.iterdir() if p.is_file()]
)

pd.DataFrame(
    {"generated_file": manifest}
).to_csv(
    OUT_DIR / "step7_output_manifest.csv",
    index=False,
)

print("=" * 80)
print("STEP 7 COMPLETED SUCCESSFULLY")
print("=" * 80)
print("Output folder:", OUT_DIR)
print("\nMost important files:")
print(" - german_ensemble_decision_table.csv")
print(" - german_ensemble_summary.csv")
print(" - german_ensemble_fold_results.csv")
print(" - german_ensemble_outer_predictions.csv")


STEP 7 COMPLETED SUCCESSFULLY
Output folder: D:\PHD\Research Paper writing\3rd Obj. paper\results\ensemble_german_development

Most important files:
 - german_ensemble_decision_table.csv
 - german_ensemble_summary.csv
 - german_ensemble_fold_results.csv
 - german_ensemble_outer_predictions.csv
